## Subset/Zoom-into the GRNs using regulatory programs

- 1) EDA on regulatory programs (peak clusters) that has interesting set of TFs and genes.
- 2) make the EDA on 1) in a systematic way (using modules and functions)
- 3) Use the module to subset the big GRN -> visualize (maybe in a separate notebook)

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy.stats import hypergeom
import sys
import os

from gimmemotifs import maelstrom

# rapids-singlecell
import cupy as cp
import rapids_singlecell as rsc

# For maelstrom module
import logging
from functools import partial
from multiprocessing import Pool
from scipy.stats import pearsonr

In [2]:
# figure parameter setting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
mpl.rcParams.update(mpl.rcParamsDefault) #Reset rcParams to default

# Editable text and proper LaTeX fonts in illustrator
# matplotlib.rcParams['ps.useafm'] = True
# Editable fonts. 42 is the magic number
mpl.rcParams['pdf.fonttype'] = 42
sns.set(style='whitegrid', context='paper')
# Set default DPI for saved figures
mpl.rcParams['savefig.dpi'] = 600

In [3]:
import logging
# Suppress INFO-level logs for the entire logger
logging.getLogger().setLevel(logging.WARNING)

In [ ]:
# import custom utils

In [5]:
# define the figure path
figpath = "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/zebrahub-multiome-analysis/figures/reg_programs/"
os.makedirs(figpath, exist_ok=True)
sc.settings.figdir = figpath

In [6]:
# import the peaks-by-celltype&timepoint pseudobulk object
adata_peaks = sc.read_h5ad("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/peaks_by_pb_leiden_0.4_merged_annotated_filtered.h5ad")
adata_peaks

AnnData object with n_obs × n_vars = 640830 × 190
    obs: 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'accessibility_neural_optic', 'accessibility_endoderm', 'accessibility_enteric_neurons', 'accessibility_notochord', 'accessibility_pharyngeal_arches', 'accessibility_floor_plate', 'accessibility_epidermis', 'accessibility_pronephros', 'accessibility_neural_floor_plate', 'accessibility_lateral_plate_mesoderm', 'accessibility_midbrain_hindbrain_boundary', 'accessibility_neural', 'accessibility_neurons', 'accessibility_neural_posterior', 'accessibility_neural_crest', 'accessibility_PSM', 'accessibility_fast_muscle', 'accessibility_endocrine_pancreas', 'accessibility_heart_myocardium', 'accessibility_neural_telencephalon', 'accessibility_optic_cup', 'accessibility_primordial_germ_cells', 'accessibility_differentiating_neurons', 'accessibility_muscle', 'accessibility_tail_bud', 'accessibility_hindbrain', 'accessibility_somites', 'accessibility_spinal_cord',

In [7]:
# import the annotation ("associated_genes")
df_genes_anno = pd.read_csv("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/all_peaks_annotated.csv",
                            index_col=0)
df_genes_anno.head()

,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,accessibility_neural_optic,accessibility_endoderm,accessibility_enteric_neurons,accessibility_notochord,accessibility_pharyngeal_arches,accessibility_floor_plate,...,is_linked,chrom,start,end,peak_type,length,gene_body_overlaps,nearest_gene,distance_to_tss,leiden_coarse
1-32-526,131,2.694737,31.052632,512.0,15.613128,10.325988,6.689899,2.991895,12.762588,12.170900,...,Not linked,1,32,526,intergenic,494,NaN,cep97,11542.0,7
1-2372-3057,162,8.294737,14.736842,1576.0,57.789060,19.729371,42.719110,17.231317,22.624311,36.413397,...,Not linked,1,2372,3057,intergenic,685,NaN,cep97,9107.0,12
1-3427-4032,170,18.636842,10.526316,3541.0,94.465606,30.845398,105.145162,41.163118,72.624480,72.500155,...,Not linked,1,3427,4032,intergenic,605,NaN,cep97,8092.0,33
1-4469-7268,190,160.247368,0.000000,30447.0,568.299814,493.321985,480.077318,535.338964,641.692987,543.858841,...,Not linked,1,4469,7268,exonic,2799,rpl24,cep97,5953.0,26
1-9541-9969,172,33.905263,9.473684,6442.0,118.378152,102.449880,65.013071,141.397103,178.130543,108.393402,...,Not linked,1,9541,9969,promoter,428,rpl24,cep97,2066.0,33


In [8]:
# copy over the metadata ("linked_gene", etc.)
metadata_list = ['linked_gene', 'link_score', 'link_zscore', 'link_pvalue']
for col in metadata_list:
    adata_peaks.obs[col] = adata_peaks.obs_names.map(df_genes_anno[col])

adata_peaks.obs["linked_gene"]


1-32-526                NaN
1-2372-3057             NaN
1-3427-4032             NaN
1-4469-7268             NaN
1-9541-9969             NaN
                       ... 
25-37496420-37496948    NaN
25-37497049-37497789    NaN
25-37498106-37500090    NaN
25-37500598-37500859    NaN
25-37501104-37501839    NaN
Name: linked_gene, Length: 640830, dtype: object

In [14]:
print(adata_peaks.obs["linked_gene"].unique()[0:10])

[nan 'rpl24' 'gpa33a' 'cenpe' 'gas6' 'zgc:92518' 'jam2a' 'appa' 'runx1'
 'atp1a1a.4']


In [23]:
# Check the example genes for each peak cluster
for cluster in adata_peaks.obs["leiden_coarse"].unique():
    # subset
    adata_sub = adata_peaks[adata_peaks.obs["leiden_coarse"]==cluster]
    num_genes = len(adata_sub.obs["linked_gene"].unique())
    # print out the example list of genes
    print(f"cluster {cluster} has {num_genes} associated genes:")
    print(adata_sub.obs["linked_gene"].unique()[1:11])

cluster 7 has 129 associated genes:
['ank2a' 'dlc1' 'tll1' 'pax5' 'sfrp2' 'slc2a15b' 'glra3' 'gata6' 'dmbx1a'
 'col11a1b']
cluster 12 has 101 associated genes:
['tll1' 'itga6b' 'slc2a15b' 'GALNTL6' 'gpm6ab' 'gfi1aa' 'col11a1b'
 'ptger4c' 'myh7l' 'cahz']
cluster 33 has 1342 associated genes:
['rpl24' 'zgc:92518' 'jam2a' 'mbnl2' 'gpc6a' 'gpc5a' 'fn1b' 'nrp2a'
 'klf7a' 'BX004779.2']
cluster 26 has 1959 associated genes:
['rpl24' 'gpa33a' 'cenpe' 'appa' 'mbnl2' 'farp1' 'gpc6a' 'tuba8l2' 'mnx2b'
 'fn1b']
cluster 11 has 301 associated genes:
['gas6' 'atp1a1a.4' 'fn1b' 'nrp2a' 'klf7a' 'BX004779.2' 'dlc1' 'mtus1b'
 'rbm47' 'pax5']
cluster 10 has 336 associated genes:
['mbnl2' 'fn1b' 'dmd' 'ank2a' 'limch1a' 'sh3d19' 'kcnq5b' 'lmo7b' 'dclk2a'
 'dok7']
cluster 23 has 218 associated genes:
['rpl24' 'dlc1' 'fat1a' 'mab21l2' 'tet2' 'si:dkey-25o16.4' 'zic2b'
 'znf827' 'si:dkey-117m1.4' 'rps6']
cluster 13 has 297 associated genes:
['pcdh18a' 'apela' 'ndst3' 'tll1' 'pax5' 'prom1b' 'bnc2' 'kcnq5b' 'dok7

In [24]:
# import the functions to annotate the peaks with "associated genes"
# this function uses "linked_genes" from Signac and "overlapping with gene body" based on GTF file
from utils_gene_annotate import *

In [25]:
# associate peaks to genes
# (1) use "linked_gene" if possible
# (2) use "gene_body_overlaps" as secondary
# (3) add NaN otherwise
adata_peaks = create_gene_associations(adata_peaks)
adata_peaks.obs.head()

,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,accessibility_neural_optic,accessibility_endoderm,accessibility_enteric_neurons,accessibility_notochord,accessibility_pharyngeal_arches,accessibility_floor_plate,...,gene_body_overlaps,nearest_gene,distance_to_tss,leiden_coarse,linked_gene,link_score,link_zscore,link_pvalue,associated_gene,association_type
1-32-526,131,2.694737,31.052632,512.0,15.613128,10.325988,6.689899,2.991895,12.762588,12.170900,...,,cep97,11542.0,7,NaN,NaN,NaN,NaN,None,none
1-2372-3057,162,8.294737,14.736842,1576.0,57.789060,19.729371,42.719110,17.231317,22.624311,36.413397,...,,cep97,9107.0,12,NaN,NaN,NaN,NaN,None,none
1-3427-4032,170,18.636842,10.526316,3541.0,94.465606,30.845398,105.145162,41.163118,72.624480,72.500155,...,,cep97,8092.0,33,NaN,NaN,NaN,NaN,None,none
1-4469-7268,190,160.247368,0.000000,30447.0,568.299814,493.321985,480.077318,535.338964,641.692987,543.858841,...,rpl24,cep97,5953.0,26,NaN,NaN,NaN,NaN,rpl24,overlap
1-9541-9969,172,33.905263,9.473684,6442.0,118.378152,102.449880,65.013071,141.397103,178.130543,108.393402,...,rpl24,cep97,2066.0,33,NaN,NaN,NaN,NaN,rpl24,overlap


In [30]:
adata_peaks.write_h5ad("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/peaks_by_pb_leiden_0.4_merged_annotated_filtered.h5ad")
print("updated object saved")

... storing 'linked_gene' as categorical
... storing 'associated_gene' as categorical


updated object saved
